# MonoSplat Colab — Setup & Patch

This notebook sets up the environment, installs a few missing dependencies, and patches `gaussian_model._knn_mean_dist` to avoid OOM on very large COLMAP point clouds (sampling + chunked lookup). Run cells in order; after the setup you can run training cells as usual.

# 🌐 MonoSplat — Gaussian Splat Training on Colab GPU
**Version 2.1 — 360GS-aligned | Object / Product / Architecture Mode**

---

## Before You Run

This notebook handles **only the GPU training stage**.  
All CPU stages (FFmpeg frame extraction + COLMAP SfM) must have already completed on your desktop.

### Prerequisites — complete ALL of these first:
1. ✅ Start the MonoSplat server locally: `uvicorn src.pipeline.server:app --reload --port 8000`
2. ✅ Upload your video at `http://localhost:8000`
3. ✅ Wait for the progress bar to complete **Frames** and **COLMAP** stages (stalling at *Train* is expected)
4. ✅ Note your `job_id` from `models/registry.json`

---

## Step 0 — Create your zip on the desktop

**Windows (PowerShell) — replace `<job_id>` with your actual job_id:**
```powershell
Compress-Archive -Path work\<job_id>\frames, work\<job_id>\colmap, config -DestinationPath monosplat_job.zip
```

**OR use the helper script (recommended):**
```powershell
python scripts/zip_for_colab.py <job_id>
```

**Mac / Linux:**
```bash
python scripts/zip_for_colab.py <job_id>
# or manually:
zip -r monosplat_job.zip work/<job_id>/frames work/<job_id>/colmap config/
```

The zip must contain:
```
monosplat_job.zip
├── work/
│   └── <job_id>/
│       ├── frames/              ← PNG frames from FFmpeg
│       └── colmap/
│           └── sparse_text/     ← cameras.txt, images.txt, points3D.txt
└── config/
    └── config.yaml
```

Then run cells **top to bottom** — do not skip any cell.

---

## After Training — Steps on your desktop

Cell 7 downloads `.splat` and `.ply` files automatically. Then on your desktop:

1. Copy files to `work/<job_id>/models/gaussian/`
2. Open `models/registry.json`, find your job, and update:
   ```json
   "status": "ready",
   "splat_path": "work/<job_id>/models/gaussian/<job_id>.splat",
   "ply_path":   "work/<job_id>/models/gaussian/<job_id>.ply"
   ```
3. Open `http://localhost:8000/viewer/<job_id>` — your 3D scene is live.

Or drag the `.splat` directly to [SuperSplat](https://supersplat.playcanvas.com) for quick preview.

---

## GPU Limits Reference

| GPU | Safe `max_gaussians` | Safe `batch_size` | Max `iterations` |
|-----|---------------------|-------------------|-----------------|
| T4 (free Colab) | 50,000 | 1,500 | 5,000 |
| V100 / A100 | 500,000 | 5,000 | 30,000 |
| CPU only | 5,000 | 500 | 500 |

Increase these values in **Cell 6** if your runtime has more VRAM.


In [1]:
# ── Cell 1: Verify GPU ─────────────────────────────────────────────────────
# You MUST see a GPU listed below.
# If not: Runtime → Change runtime type → T4 GPU → Save → then re-run all cells.

import subprocess, sys, os

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)

if result.returncode != 0:
    print("❌  No GPU found!")
    print("   → Runtime → Change runtime type → T4 GPU → Save → Reconnect")
    raise SystemExit("No GPU detected. Change runtime type first.")

print("✅  GPU detected:")
for line in result.stdout.split("\n")[:13]:
    print(line)

# Also confirm we are on Python 3.9+
assert sys.version_info >= (3, 9), f"Python 3.9+ required, got {sys.version}"
print(f"\n✅  Python {sys.version.split()[0]}")

# Set memory allocator env var BEFORE importing torch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("✅  PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True")


✅  GPU detected:
Sun May 17 08:01:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

In [2]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
# PyTorch is pre-installed on Colab. We add only the extras needed.
# Runtime: ~30 seconds.

import subprocess, sys

def pip_install(*packages):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        stdout=subprocess.DEVNULL
    )

pip_install("pillow", "pyyaml", "plyfile", "tqdm", "numpy")

# Optional: install lpips for perceptual loss (adds ~60 s, skipped by default).
# Uncomment the line below if you want LPIPS-augmented training:
# pip_install("lpips")

import torch
print(f"✅  PyTorch   : {torch.__version__}")
print(f"   CUDA       : {torch.version.cuda}")
print(f"   Available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1e9
    print(f"   GPU        : {props.name}")
    print(f"   VRAM       : {vram:.1f} GB")
    if vram < 12:
        print(f"\n⚠️  Less than 12 GB VRAM — keep max_gaussians ≤ 50,000 and batch_size ≤ 1,500")
    else:
        print(f"\n✅  Plenty of VRAM — you can increase max_gaussians and iterations in Cell 6")
else:
    print("❌  CUDA not available — check runtime type.")
    raise SystemExit("CUDA not available.")

# Clear any residual GPU memory from previous runs
torch.cuda.empty_cache()
print("\n✅  GPU memory cleared. Ready for upload.")


✅  PyTorch   : 2.10.0+cu128
   CUDA       : 12.8
   Available  : True
   GPU        : Tesla T4
   VRAM       : 15.6 GB

✅  Plenty of VRAM — you can increase max_gaussians and iterations in Cell 6

✅  GPU memory cleared. Ready for upload.


In [3]:
# ── Cell 2.5: Install GPU rasterizer ──────────────────────────────────────
# REQUIRED — without this the renderer falls back to a Python loop
# that takes ~20s/iter instead of <1s/iter.
# Runtime: <1 minute when a wheel is available.

import subprocess, sys, torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
print(f"cuDNN   : {torch.backends.cudnn.version()}")
print()

# Install gsplat for the fast CUDA rasterizer path used by this repo.
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gsplat"])
    print("✅  gsplat install command completed.")
except subprocess.CalledProcessError as e:
    raise RuntimeError(
        "❌  Failed to install gsplat. "
        "Try: Runtime → Factory reset runtime → Run all from the top.\n"
        f"Command error: {e}"
    )

# ── Final verification ──────────────────────────────────────────────────────
try:
    import importlib
    importlib.invalidate_caches()
    gsplat = importlib.import_module("gsplat")
    print(f"\n✅  gsplat imported successfully. version={getattr(gsplat, '__version__', 'unknown')}")
    print("\n→  Run Cell 3 to upload your zip.")
except ImportError as e:
    raise ImportError(
        f"❌  gsplat import failed after install: {e}\n"
        "   Try: Runtime → Restart runtime → Run all from the top."
    )


PyTorch : 2.10.0+cu128
CUDA    : 12.8
cuDNN   : 91002

✅  gsplat install command completed.

✅  gsplat imported successfully. version=1.5.3

→  Run Cell 3 to upload your zip.


In [4]:
# ── Cell 3: Upload monosplat_job.zip ───────────────────────────────────────
from google.colab import files
import zipfile, os, glob, shutil

# ── FIX: force working directory to /content before upload ───────────────────
# files.upload() saves to cwd — if cwd doesn't exist or changed, it crashes.
os.chdir("/content")
os.makedirs("/content/monosplat", exist_ok=True)
print(f"Working directory set to: {os.getcwd()}")

print("\n📂  Select your monosplat_job.zip when the file picker opens...")
print("    (Upload time depends on file size — frames folder can be 100–300 MB)")
print()

uploaded = files.upload()

if not uploaded:
    raise RuntimeError(
        "No file uploaded.\n"
        "Please upload monosplat_job.zip and re-run this cell."
    )

zip_name  = list(uploaded.keys())[0]
zip_size  = os.path.getsize(zip_name) / 1e6
print(f"\n📦  Uploaded : {zip_name}  ({zip_size:.1f} MB)")

# Move zip into /content/monosplat and extract there
zip_src  = os.path.join("/content", zip_name)
zip_dest = os.path.join("/content/monosplat", zip_name)
shutil.move(zip_src, zip_dest)

EXTRACT_DIR = "/content/monosplat"
print(f"📂  Extracting to {EXTRACT_DIR} ...")
with zipfile.ZipFile(zip_dest, "r") as z:
    z.extractall(EXTRACT_DIR)

os.chdir(EXTRACT_DIR)
print(f"📂  Working directory: {os.getcwd()}")

# ── Auto-detect job_id ────────────────────────────────────────────────────────
work_dirs = [d for d in glob.glob("work/*") if os.path.isdir(d)]
if not work_dirs:
    raise RuntimeError(
        "\n❌  Could not find work/<job_id>/ inside the zip.\n"
        "   The zip must contain:\n"
        "     work/<job_id>/frames/\n"
        "     work/<job_id>/colmap/sparse_text/\n"
        "     config/config.yaml\n"
        "   Re-create the zip with: python scripts/zip_for_colab.py <job_id>"
    )

JOB_ID      = os.path.basename(sorted(work_dirs)[0])
FRAMES_DIR  = f"work/{JOB_ID}/frames"
COLMAP_DIR  = f"work/{JOB_ID}/colmap/sparse_text"
OUTPUT_DIR  = f"work/{JOB_ID}/models/gaussian"
CKPT_DIR    = f"work/{JOB_ID}/models/checkpoints"
CONFIG_PATH = "config/config.yaml"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR,   exist_ok=True)

print(f"\n✅  Job ID     : {JOB_ID}")
print(f"   Frames dir : {FRAMES_DIR}  ({len(glob.glob(FRAMES_DIR + '/*'))} files)")
print(f"   COLMAP dir : {COLMAP_DIR}")
print(f"   Output dir : {OUTPUT_DIR}")
print(f"   Config     : {CONFIG_PATH}")
print(f"\n→  Run Cell 4 to verify COLMAP files.")

Working directory set to: /content

📂  Select your monosplat_job.zip when the file picker opens...
    (Upload time depends on file size — frames folder can be 100–300 MB)



Saving 2e9c989b96ce_for_colab.zip to 2e9c989b96ce_for_colab.zip

📦  Uploaded : 2e9c989b96ce_for_colab.zip  (100.9 MB)
📂  Extracting to /content/monosplat ...
📂  Working directory: /content/monosplat

✅  Job ID     : 2e9c989b96ce
   Frames dir : work/2e9c989b96ce/frames  (197 files)
   COLMAP dir : work/2e9c989b96ce/colmap/sparse_text
   Output dir : work/2e9c989b96ce/models/gaussian
   Config     : config/config.yaml

→  Run Cell 4 to verify COLMAP files.


In [5]:
# ── Cell 4: Verify COLMAP output ───────────────────────────────────────────
# All three files must exist AND have data lines.
# If any are missing → COLMAP did not finish on your desktop.
# Fix: go back to http://localhost:8000 and check the COLMAP stage.
#
# Common causes of COLMAP failure:
#   - Not enough image overlap (record slower, closer to object)
#   - Too much motion blur
#   - Reflective / transparent surfaces
#   - Fewer than ~30 frames (increase video_fps in config.yaml)

import os, glob

print(f"Checking COLMAP output in: {COLMAP_DIR}\n")
all_ok = True

for fname in ["cameras.txt", "images.txt", "points3D.txt"]:
    path = os.path.join(COLMAP_DIR, fname)
    if os.path.exists(path):
        with open(path) as f:
            data_lines = [l for l in f if not l.startswith("#") and l.strip()]
        status = "✅" if data_lines else "⚠️  EMPTY"
        print(f"  {status}  {fname:<20}  {len(data_lines):>6} data lines")
        if not data_lines:
            all_ok = False
    else:
        print(f"  ❌  {fname:<20}  MISSING")
        all_ok = False

# Count and validate frames
frame_files = (
    glob.glob(os.path.join(FRAMES_DIR, "*.png")) +
    glob.glob(os.path.join(FRAMES_DIR, "*.jpg")) +
    glob.glob(os.path.join(FRAMES_DIR, "*.jpeg"))
)
print(f"\n  📷  Frames found : {len(frame_files)}")

if not all_ok:
    print("\n❌  COLMAP is incomplete. Do NOT proceed with training.")
    print("   → Complete the COLMAP stage on your desktop first.")
elif len(frame_files) == 0:
    print("\n❌  No frames found. Check your zip — frames/ may be empty.")
    all_ok = False
elif len(frame_files) < 20:
    print(f"\n⚠️  Only {len(frame_files)} frames — reconstruction may be sparse.")
    print("   Consider increasing video_fps in config.yaml and re-running FFmpeg.")
else:
    print(f"\n✅  All COLMAP files present. {len(frame_files)} frames ready.")
    print("   → Run Cell 5 to fetch source code.")

if not all_ok:
    raise SystemExit("COLMAP verification failed — fix issues before continuing.")


Checking COLMAP output in: work/2e9c989b96ce/colmap/sparse_text

  ✅  cameras.txt                1 data lines
  ✅  images.txt               394 data lines
  ✅  points3D.txt          126394 data lines

  📷  Frames found : 197

✅  All COLMAP files present. 197 frames ready.
   → Run Cell 5 to fetch source code.


In [6]:
# ── Cell 5: Fetch src/ and scripts/ from GitHub ────────────────────────────
# Always does a clean re-clone so you get the latest bug fixes automatically.
# No need to re-upload the notebook when code is updated on GitHub.
#
# If the repo is private, replace the clone URL with your token URL:
#   https://<TOKEN>@github.com/Somaskandan931/monosplat.git

import os, sys, subprocess, shutil

REPO_URL  = "https://github.com/Somaskandan931/monosplat.git"
TMP_DIR   = "/tmp/monosplat_src"
CWD       = os.getcwd()   # /content/monosplat

src_dir     = os.path.join(CWD, "src")
scripts_dir = os.path.join(CWD, "scripts")

# ── Clean previous fetches ──────────────────────────────────────────────────
for d in [TMP_DIR, src_dir, scripts_dir]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"🗑️   Removed: {d}")

# ── Sparse-clone only src/ and scripts/ (much faster than full clone) ──────
print(f"\n📦  Cloning {REPO_URL} ...")
r = subprocess.run(
    ["git", "clone", "--filter=blob:none", "--sparse", REPO_URL, TMP_DIR],
    capture_output=True, text=True
)
if r.returncode != 0:
    print("STDERR:", r.stderr[-2000:])
    raise RuntimeError(
        "❌  Git clone failed.\n"
        "   Is https://github.com/Somaskandan931/monosplat public?\n"
        "   Check your internet connection and repo visibility."
    )

r2 = subprocess.run(
    ["git", "-C", TMP_DIR, "sparse-checkout", "set", "src", "scripts"],
    capture_output=True, text=True
)
if r2.returncode != 0:
    print("STDERR:", r2.stderr)
    raise RuntimeError("❌  sparse-checkout failed.")

# ── Validate clone succeeded ────────────────────────────────────────────────
if not os.path.isdir(os.path.join(TMP_DIR, "src")):
    raise RuntimeError(
        "❌  src/ not found in cloned repo.\n"
        "   Check that src/ exists on the main branch of the repo."
    )

# ── Copy into working directory ─────────────────────────────────────────────
shutil.copytree(os.path.join(TMP_DIR, "src"),     src_dir)
shutil.copytree(os.path.join(TMP_DIR, "scripts"), scripts_dir, dirs_exist_ok=True)
print(f"✅  src/ copied to     : {src_dir}")
print(f"✅  scripts/ copied to : {scripts_dir}")

# ── Add CWD to sys.path so imports work ─────────────────────────────────────
if CWD not in sys.path:
    sys.path.insert(0, CWD)
    print(f"✅  sys.path prepended : {CWD}")

# ── Verify critical imports ─────────────────────────────────────────────────
try:
    from src.utils.config_loader    import load_config
    from src.preprocessing.utils    import load_colmap_model
    from src.reconstruction.gaussian_model import GaussianModel
    from src.reconstruction.trainer import GaussianTrainer
    from src.reconstruction.loss    import combined_loss, psnr_metric, ssim_metric
    from src.renderer.camera        import Camera as ViewerCamera
    from src.renderer.renderer      import GaussianRenderer
    print("✅  All src/ imports successful.")
except ImportError as e:
    raise ImportError(
        f"❌  Import failed: {e}\n"
        "   Check that src/ was fetched correctly (re-run this cell)."
    )

print("\n→  Run Cell 6 to configure and start training.")


🗑️   Removed: /content/monosplat/src
🗑️   Removed: /content/monosplat/scripts

📦  Cloning https://github.com/Somaskandan931/monosplat.git ...
✅  src/ copied to     : /content/monosplat/src
✅  scripts/ copied to : /content/monosplat/scripts
✅  sys.path prepended : /content/monosplat
✅  All src/ imports successful.

→  Run Cell 6 to configure and start training.


In [7]:
# ── Cell 6: Training with Quality Settings ─────────────────────────────────────
# This cell runs the full training loop with updated quality settings:
# - 15,000 iterations (5× more refinement)
# - 100,000 Gaussians (3.3× more detail)
# - 960×540 resolution (4× more pixels)
# - SH degree 3 (full view-dependent appearance)
#
# Uses gsplat CUDA rasterizer (installed in Cell 2.5) for fast rendering.

import os, sys, gc, functools, warnings
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()

# ── Imports ──────────────────────────────────────────────────────────────────
from src.utils.config_loader            import load_config
from src.preprocessing.utils           import load_colmap_model
from src.reconstruction.gaussian_model  import GaussianModel
from src.reconstruction.trainer        import GaussianTrainer
from src.reconstruction                import gaussian_model as _gm
from src.reconstruction.loss           import combined_loss
from src.renderer.camera               import Camera as ViewerCamera
from src.renderer.renderer             import GaussianRenderer

# ── Patch A: robust kNN (prevents crash on sparse clouds) ────────────────────
def _knn_robust(pts, k=3):
    N = pts.shape[0]
    if N == 0:
        return torch.tensor([], device=pts.device)
    k2 = min(k, N - 1)
    if k2 == 0:
        return torch.zeros(N, device=pts.device)
    D = torch.cdist(pts, pts)
    D.fill_diagonal_(float("inf"))
    return torch.topk(D, k=k2, largest=False, dim=1).values.mean(dim=1).clamp(min=1e-7)

if hasattr(_gm, "_knn_mean_dist"):
    _gm._knn_mean_dist = _knn_robust
    print("✅  Patched: _knn_mean_dist")

# ── Patch B: densify/prune ops under no_grad ─────────────────────────────────
def _no_grad(fn):
    @functools.wraps(fn)
    def wrap(*a, **kw):
        with torch.no_grad():
            return fn(*a, **kw)
    return wrap

for _m in ("reset_opacity", "densify_and_prune", "prune_points",
           "densify_and_clone", "densify_and_split"):
    if hasattr(GaussianModel, _m):
        setattr(GaussianModel, _m, _no_grad(getattr(GaussianModel, _m)))
        print(f"✅  Patched: GaussianModel.{_m}")

# ── GPU profile with QUALITY settings ─────────────────────────────────────────
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nGPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {VRAM_GB:.1f} GB")

if VRAM_GB >= 35:                   # A100 / V100
    ITERS      = 30_000
    MAX_GAUSS  = 500_000
    SH         = 3
    W, H       = 960, 540
    BS         = 5_000
    D_FROM     = 500
    D_UNTIL    = 12_000
    D_INTERVAL = 100
    GRAD_THR   = 0.0002
elif VRAM_GB >= 12:                 # T4 free Colab — QUALITY settings
    ITERS      = 15_000
    MAX_GAUSS  = 100_000
    SH         = 3
    W, H       = 960, 540
    BS         = 3_000
    D_FROM     = 500
    D_UNTIL    = 12_000
    D_INTERVAL = 100
    GRAD_THR   = 0.0002
else:                               # CPU / very low VRAM
    ITERS      = 500
    MAX_GAUSS  = 5_000
    SH         = 0
    W, H       = 320, 240
    BS         = 300
    D_FROM     = 200
    D_UNTIL    = 400
    D_INTERVAL = 50
    GRAD_THR   = 0.0005

# ── Apply to config ───────────────────────────────────────────────────────────
cfg = load_config(CONFIG_PATH)
cfg.training.iterations             = ITERS
cfg.renderer.max_gaussians          = MAX_GAUSS
cfg.renderer.sh_degree              = SH
cfg.training.output_dir             = OUTPUT_DIR
cfg.training.checkpoint_dir         = CKPT_DIR
cfg.training.densify_from_iter      = D_FROM
cfg.training.densify_until_iter     = D_UNTIL
cfg.training.densification_interval = D_INTERVAL
cfg.training.densify_grad_threshold = GRAD_THR

# Guard: cfg.viewer may not exist if viewer: block is absent from config.yaml
if not hasattr(cfg, "viewer"):
    cfg.viewer = type("Config", (), {})()
cfg.viewer.window_width  = W
cfg.viewer.window_height = H

# Guard: save_every may be absent from config.yaml
save_every = getattr(cfg.training, "save_every", 500)

print(f"\nTraining config:")
print(f"  iterations    : {ITERS:,}")
print(f"  max_gaussians : {MAX_GAUSS:,}  ← quality settings")
print(f"  sh_degree     : {SH}")
print(f"  resolution    : {W}x{H}")
print(f"  batch_size    : {BS}")
print(f"  densify       : iter {D_FROM} → {D_UNTIL}, every {D_INTERVAL} iters")
print(f"  grad_threshold: {GRAD_THR}")
print(f"  save_every    : {save_every}")

# ── Load COLMAP ───────────────────────────────────────────────────────────────
print(f"\n📂  Loading COLMAP from: {COLMAP_DIR}")
cameras_colmap, images_colmap, points3d = load_colmap_model(COLMAP_DIR)

# ── Build training cameras + images ───────────────────────────────────────────
print(f"🖼️   Loading {len(images_colmap)} images at {W}x{H}...")
train_cameras, train_images, missing = [], [], []

for img_data in sorted(images_colmap.values(), key=lambda i: i.name):
    cam_data = cameras_colmap[img_data.camera_id]
    cam      = ViewerCamera.from_colmap(img_data, cam_data, W, H)
    img_path = Path(FRAMES_DIR) / img_data.name
    if not img_path.exists():
        img_path = Path(FRAMES_DIR) / Path(img_data.name).name
    if not img_path.exists():
        missing.append(img_data.name)
        continue
    img_np = np.array(
        Image.open(img_path).convert("RGB").resize((W, H), Image.LANCZOS),
        dtype=np.float32
    ) / 255.0
    train_cameras.append(cam)
    train_images.append(torch.from_numpy(img_np).permute(2, 0, 1).cpu())

if missing:
    print(f"⚠️  {len(missing)} frames missing — first 3: {missing[:3]}")
if not train_cameras:
    raise RuntimeError(f"No images loaded. Check FRAMES_DIR={FRAMES_DIR}")
print(f"✅  {len(train_cameras)} cameras, {len(points3d):,} 3D points")

# ── Subsample initial points if above MAX_GAUSS ───────────────────────────────
xyzs = np.array([p.xyz for p in points3d.values()], dtype=np.float32)
rgbs = np.array([p.rgb for p in points3d.values()], dtype=np.float32) / 255.0
if len(xyzs) > MAX_GAUSS:
    idx  = np.random.choice(len(xyzs), MAX_GAUSS, replace=False)
    xyzs = xyzs[idx]
    rgbs = rgbs[idx]
    print(f"⚠️  Initial cloud subsampled: {len(points3d):,} → {MAX_GAUSS:,} points")

# ── Init model ────────────────────────────────────────────────────────────────
device = "cuda"
model = GaussianModel(sh_degree=SH)
model = model.to(device)
model.create_from_points(xyzs, rgbs)

# ── Renderer ──────────────────────────────────────────────────────────────────
torch.cuda.empty_cache()
gc.collect()
torch.backends.cudnn.benchmark = True

renderer = GaussianRenderer(
    width=W, height=H,
    bg_color=cfg.renderer.background_color,
    device=device,
    batch_size=BS,
    use_gsplat=True,
)

# ── Trainer ───────────────────────────────────────────────────────────────────
trainer = GaussianTrainer(
    model=model,
    renderer=renderer.render_torch,
    train_cameras=train_cameras,
    train_images=train_images,
    cfg=cfg,
)

# ── VRAM estimate ─────────────────────────────────────────────────────────────
bytes_per_gauss = (3 + 1 + 3 + 3 + 4 + 48) * 4
vram_est_mb     = len(model) * bytes_per_gauss / 1e6
free_vram_gb    = torch.cuda.mem_get_info()[0] / 1e9
print(f"\n📊  VRAM estimate : ~{vram_est_mb:.0f} MB for {len(model):,} Gaussians")
print(f"    Free VRAM now : {free_vram_gb:.1f} GB / {VRAM_GB:.1f} GB total")
if vram_est_mb > free_vram_gb * 1000 * 0.8:
    print("⚠️  Close to VRAM limit — lower MAX_GAUSS if you get OOM")

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  MonoSplat — Quality Training")
print(f"  Job : {JOB_ID}  |  GPU : {torch.cuda.get_device_name(0)}")
print(f"  {ITERS:,} iters  |  max {MAX_GAUSS:,} Gaussians")
print(f"{'='*60}\n")

optimizer     = trainer.optimizer
n             = len(model)
grad_accum    = torch.zeros(n, device=device)
grad_denom    = torch.zeros(n, device=device)
nan_count     = 0
oom_count     = 0
loss_val      = 0.0
lambda_dssim  = getattr(cfg.training, "lambda_dssim", 0.2)
percent_dense = getattr(cfg.training, "percent_dense", 0.01)

pbar = tqdm(range(ITERS), desc="Training", dynamic_ncols=True)

for it in pbar:
    idx    = np.random.randint(len(train_cameras))
    camera = train_cameras[idx]
    gt_img = train_images[idx].to(device, non_blocking=True)

    try:
        optimizer.zero_grad(set_to_none=True)
        rendered = renderer.render_torch(model, camera)
        loss     = combined_loss(rendered, gt_img, lambda_ssim=lambda_dssim)

        if torch.isnan(loss):
            nan_count += 1
            if nan_count <= 5 or nan_count % 50 == 0:
                tqdm.write(f"[{it:5d}] NaN loss (total {nan_count}), skipping")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        if model._positions.grad is not None:
            with torch.no_grad():
                gn = model._positions.grad.norm(dim=1)
                if gn.shape[0] == grad_accum.shape[0]:
                    grad_accum += gn
                    grad_denom += 1.0

        optimizer.step()
        trainer._apply_position_lr_decay(it, ITERS)
        loss_val = loss.item()

    except torch.cuda.OutOfMemoryError:
        oom_count += 1
        torch.cuda.empty_cache()
        gc.collect()
        tqdm.write(f"[{it:5d}] OOM #{oom_count} — emergency prune to 60% cap")
        emergency = int(MAX_GAUSS * 0.6)
        cur = len(model)
        if cur > emergency:
            n_prune = cur - emergency
            with torch.no_grad():
                _, low = model.opacities.squeeze().topk(n_prune, largest=False)
                mask   = torch.zeros(cur, dtype=torch.bool, device=device)
                mask[low] = True
            model.prune_points(mask)
            trainer._setup_optimizer()
            optimizer  = trainer.optimizer
            n          = len(model)
            grad_accum = torch.zeros(n, device=device)
            grad_denom = torch.zeros(n, device=device)
        continue

    # SH degree scheduling
    if it % 1000 == 0:
        model.oneup_sh_degree()

    # Densification
    if D_FROM <= it <= D_UNTIL and it % D_INTERVAL == 0:
        trainer._grad_accum = grad_accum.clone()
        trainer._grad_denom = grad_denom.clone()
        trainer._densify_and_prune(grad_threshold=GRAD_THR, percent_dense=percent_dense)
        optimizer  = trainer.optimizer
        n          = len(model)
        grad_accum = torch.zeros(n, device=device)
        grad_denom = torch.zeros(n, device=device)
        tqdm.write(f"[{it:5d}] Densified -> {n:,} Gaussians")

    # Checkpoint
    if it > 0 and it % save_every == 0:
        trainer._save(it)

    # VRAM periodic clear
    if it % 500 == 0:
        torch.cuda.empty_cache()

    pbar.set_postfix({
        "loss": f"{loss_val:.4f}",
        "N":    f"{len(model):,}",
        "NaN":  nan_count,
        "OOM":  oom_count,
    })

trainer._save(ITERS)

print(f"\n✅  Training complete.")
print(f"   Gaussians   : {len(model):,}")
print(f"   Final loss  : {loss_val:.4f}")
print(f"   NaN skipped : {nan_count}")
print(f"   OOM events  : {oom_count}")
print(f"   Output dir  : {OUTPUT_DIR}")
print(f"\n→  Run Cell 7 to export and download your .splat and .ply files.")


Build artifacts in /tmp/dgr:
  (nothing — /tmp/dgr does not exist)

pip show:
  (not installed)

❌  Import failed: No module named 'diff_gaussian_rasterization'


In [13]:
# ── Cell 7: Export and Download .splat and .ply ───────────────────────────────
# This cell exports the trained model to browser-ready .splat format
# and downloads both .splat and .ply files to your local machine.

import glob
from google.colab import files
from src.utils.io_utils import save_splat, load_ply

print("\n" + "="*60)
print("  Exporting .splat and .ply")
print("="*60)

# Find the latest .ply checkpoint
ply_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.ply")), key=os.path.getmtime)
if not ply_files:
    raise RuntimeError(f"No .ply files found in {OUTPUT_DIR}")

latest_ply = ply_files[-1]
print(f"Latest PLY : {latest_ply}  ({os.path.getsize(latest_ply)/1e6:.1f} MB)")

# Load and export as .splat
try:
    gaussians = load_ply(latest_ply)
    splat_path = os.path.join(OUTPUT_DIR, f"{JOB_ID}.splat")
    save_splat(splat_path, gaussians)
    print(f"Exported   : {splat_path}  ({os.path.getsize(splat_path)/1e6:.1f} MB)")
except Exception as e:
    print(f".splat export failed: {e}")
    print("You can still use the .ply file.")
    splat_path = None

# Copy .ply to named file
ply_dest = os.path.join(OUTPUT_DIR, f"{JOB_ID}.ply")
if os.path.abspath(latest_ply) != os.path.abspath(ply_dest):
    import shutil
    shutil.copy2(latest_ply, ply_dest)
print(f"PLY ready  : {ply_dest}")

# Download files
print("\n" + "="*60)
print("  Downloading files...")
print("="*60)

files_to_download = [ply_dest]
if splat_path and os.path.exists(splat_path):
    files_to_download.append(splat_path)

for fpath in files_to_download:
    print(f"\nDownloading: {os.path.basename(fpath)}")
    files.download(fpath)

print("\n" + "="*60)
print("  Done! Next steps on your desktop:")
print("="*60)
print(f"1. Copy downloaded files to: work/{JOB_ID}/models/gaussian/")
print(f"2. Edit models/registry.json:")
print(f'   "status": "ready",')
print(f'   "splat_path": "work/{JOB_ID}/models/gaussian/{JOB_ID}.splat",')
print(f'   "ply_path": "work/{JOB_ID}/models/gaussian/{JOB_ID}.ply"')
print(f"3. Restart the server:")
print(f"   uvicorn src.pipeline.server:app --reload --port 8000")
print(f"4. View at: http://localhost:8000/viewer/{JOB_ID}")
print("="*60)


✅  Patched: _knn_mean_dist
✅  Patched: GaussianModel.prune_points
✅  Patched: GaussianModel.densify_and_clone
✅  Patched: GaussianModel.densify_and_split

GPU  : Tesla T4
VRAM : 15.6 GB

Training config:
  iterations    : 3,000
  max_gaussians : 30,000  ← hard cap prevents OOM
  sh_degree     : 1
  resolution    : 480x270
  batch_size    : 1000
  densify       : iter 500 → 2000, every 150 iters
  grad_threshold: 0.0003
  save_every    : 5000

📂  Loading COLMAP from: work/2e9c989b96ce/colmap/sparse_text
[colmap] Loaded model: 1 cameras, 197 images, 126,394 3D points
🖼️   Loading 197 images at 480x270...
✅  197 cameras, 126,394 3D points
⚠️  Initial cloud subsampled: 126,394 → 30,000 points
[GaussianModel] Initialised 30,000 Gaussians on cpu from point cloud.
[Renderer] ✓ Using gsplat CUDA rasterizer (fast path)
           gsplat version: 1.5.3
[Trainer] Device  : cuda
[Trainer] Backend : gsplat
[Trainer] Output  : /content/monosplat/work/2e9c989b96ce/models/gaussian

📊  VRAM estimate : 

Training:   0%|          | 0/3000 [00:00<?, ?it/s]


AssertionError: torch.Size([30000, 3])

In [ ]:
# ── Cell 7: Export .splat and Download Files ───────────────────────────────
# Converts the best .ply checkpoint to browser-ready .splat binary format,
# then downloads both files to your computer.
#
# .splat → Three.js browser viewer (http://localhost:8000/viewer/<job_id>)
# .ply   → SuperSplat editor       (https://supersplat.playcanvas.com)

import os, glob, shutil
from pathlib import Path
from google.colab import files

# ── Find the latest checkpoint .ply ─────────────────────────────────────────
ply_files = sorted(
    glob.glob(os.path.join(OUTPUT_DIR, "*.ply")),
    key=os.path.getmtime
)

if not ply_files:
    print("❌  No .ply files found.")
    print(f"   Expected them in: {OUTPUT_DIR}")
    print("   Make sure Cell 6 completed successfully.")
    raise SystemExit("No output files.")

latest_ply = ply_files[-1]
print(f"📄  Latest checkpoint : {latest_ply}")
print(f"    Size              : {os.path.getsize(latest_ply)/1e6:.1f} MB")

# ── Export .splat using io_utils ────────────────────────────────────────────
try:
    from src.utils.io_utils import save_splat, load_ply
    print("\n🔄  Converting .ply → .splat ...")
    gaussians  = load_ply(latest_ply)
    splat_path = os.path.join(OUTPUT_DIR, f"{JOB_ID}.splat")
    save_splat(splat_path, gaussians)
    print(f"✅  Exported: {splat_path}  ({os.path.getsize(splat_path)/1e6:.1f} MB)")
except Exception as e:
    print(f"⚠️  .splat export failed: {e}")
    print("   You can still use the .ply file — drag it to SuperSplat.")
    splat_path = None

# ── Copy .ply with job_id name ───────────────────────────────────────────────
ply_dest = os.path.join(OUTPUT_DIR, f"{JOB_ID}.ply")
if os.path.abspath(latest_ply) != os.path.abspath(ply_dest):
    shutil.copy2(latest_ply, ply_dest)
print(f"✅  PLY ready   : {ply_dest}  ({os.path.getsize(ply_dest)/1e6:.1f} MB)")

# ── Download files ───────────────────────────────────────────────────────────
downloaded = []

if splat_path and os.path.exists(splat_path):
    print(f"\n⬇️   Downloading {JOB_ID}.splat ...")
    files.download(splat_path)
    downloaded.append(f"{JOB_ID}.splat")

print(f"⬇️   Downloading {JOB_ID}.ply ...")
files.download(ply_dest)
downloaded.append(f"{JOB_ID}.ply")

# ── Print next-step instructions ─────────────────────────────────────────────
print(f"\n✅  Downloaded: {', '.join(downloaded)}")
print()
print("─" * 62)
print("NEXT STEPS — on your desktop:")
print("─" * 62)
print(f"  1.  Create this folder if it does not exist:")
print(f"        work\\{JOB_ID}\\models\\gaussian\\   (Windows)")
print(f"        work/{JOB_ID}/models/gaussian/    (Mac/Linux)")
print()
print(f"  2.  Move downloaded files there:")
print(f"        {JOB_ID}.splat  →  work/{JOB_ID}/models/gaussian/")
print(f"        {JOB_ID}.ply    →  work/{JOB_ID}/models/gaussian/")
print()
print(f"  3.  Open models/registry.json, find job {JOB_ID}, update:")
print(f'        "status":     "ready",')
print(f'        "splat_path": "work/{JOB_ID}/models/gaussian/{JOB_ID}.splat",')
print(f'        "ply_path":   "work/{JOB_ID}/models/gaussian/{JOB_ID}.ply"')
print()
print(f"  4.  Make sure your local server is running:")
print(f"        uvicorn src.pipeline.server:app --reload --port 8000")
print()
print(f"  5.  Open in browser:")
print(f"        http://localhost:8000/viewer/{JOB_ID}")
print()
print("  Or drag the .splat to: https://supersplat.playcanvas.com")
print("─" * 62)


In [ ]:
# ── Cell 8 (Optional): Preview Render ──────────────────────────────────────
# Renders a single frame from the trained model and displays it inline.
# Useful to quickly check reconstruction quality before downloading.
# Skip this cell if you just want the files.

import glob, os, torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage

print("🎨  Rendering preview frame from trained model...")

if "model" not in dir() or len(model) == 0:
    print("❌  No trained model in memory. Run Cell 6 first.")
else:
    # Use the middle camera for a centred view
    idx    = len(train_cameras) // 2
    camera = train_cameras[idx]

    with torch.no_grad():
        rendered = renderer.render_torch(model, camera)

    img_np = rendered.detach().cpu().permute(1, 2, 0).numpy()
    img_np = (img_np * 255).clip(0, 255).astype(np.uint8)

    # Also show the ground-truth frame for comparison
    gt_np = train_images[idx].permute(1, 2, 0).numpy()
    gt_np = (gt_np * 255).clip(0, 255).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(gt_np);     axes[0].set_title("Ground Truth");  axes[0].axis("off")
    axes[1].imshow(img_np);    axes[1].set_title("Rendered Splat"); axes[1].axis("off")
    plt.suptitle(f"MonoSplat Preview — Job {JOB_ID} — Camera {idx}", fontsize=12)
    plt.tight_layout()
    plt.show()

    # Save preview PNG to output directory
    preview_path = os.path.join(OUTPUT_DIR, f"preview_final.png")
    PILImage.fromarray(img_np).save(preview_path)
    print(f"✅  Preview saved: {preview_path}")
    print(f"   Gaussians in scene : {len(model):,}")
    print(f"   Camera             : {camera}")


In [ ]:
# Install helper packages (best-effort, non-Torch packages)
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
# common helpers used by the repo
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plyfile', 'scikit-learn'])
# faiss is optional; try cpu wheel then fall back to skipping
try:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'faiss-cpu'])
except Exception:
    print('faiss-cpu not available; continuing without it')

In [ ]:
import os, sys, gc, functools, warnings
import numpy as np
import torch
from pathlib import Path
from PIL import Image
from tqdm import tqdm

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()

# ── Imports ──────────────────────────────────────────────────────────────────
from src.utils.config_loader            import load_config
from src.preprocessing.utils           import load_colmap_model
from src.reconstruction.gaussian_model  import GaussianModel
from src.reconstruction.trainer        import GaussianTrainer
from src.reconstruction                import gaussian_model as _gm
from src.reconstruction.loss           import combined_loss
from src.renderer.camera               import Camera as ViewerCamera
from src.renderer.renderer             import GaussianRenderer

# ── Patch A: robust kNN (prevents crash on sparse clouds) ────────────────────
def _knn_robust(pts, k=3):
    N = pts.shape[0]
    if N == 0:
        return torch.tensor([], device=pts.device)
    k2 = min(k, N - 1)
    if k2 == 0:
        return torch.zeros(N, device=pts.device)
    D = torch.cdist(pts, pts)
    D.fill_diagonal_(float("inf"))
    return torch.topk(D, k=k2, largest=False, dim=1).values.mean(dim=1).clamp(min=1e-7)

if hasattr(_gm, "_knn_mean_dist"):
    _gm._knn_mean_dist = _knn_robust
    print("✅  Patched: _knn_mean_dist")

# ── Patch B: densify/prune ops under no_grad ─────────────────────────────────
def _no_grad(fn):
    @functools.wraps(fn)
    def wrap(*a, **kw):
        with torch.no_grad():
            return fn(*a, **kw)
    return wrap

for _m in ("reset_opacity", "densify_and_prune", "prune_points",
           "densify_and_clone", "densify_and_split"):
    if hasattr(GaussianModel, _m):
        setattr(GaussianModel, _m, _no_grad(getattr(GaussianModel, _m)))
        print(f"✅  Patched: GaussianModel.{_m}")

# ── GPU profile ───────────────────────────────────────────────────────────────
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nGPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {VRAM_GB:.1f} GB")

if VRAM_GB >= 35:                   # A100 / V100
    ITERS      = 30_000
    MAX_GAUSS  = 500_000
    SH         = 3
    W, H       = 960, 540
    BS         = 5_000
    D_FROM     = 500
    D_UNTIL    = 12_000
    D_INTERVAL = 100
    GRAD_THR   = 0.0002
elif VRAM_GB >= 12:                 # T4 free Colab — quality settings
    ITERS      = 15_000
    MAX_GAUSS  = 100_000
    SH         = 3
    W, H       = 960, 540
    BS         = 3_000
    D_FROM     = 500
    D_UNTIL    = 12_000
    D_INTERVAL = 100
    GRAD_THR   = 0.0002
else:                               # CPU / very low VRAM
    ITERS      = 500
    MAX_GAUSS  = 5_000
    SH         = 0
    W, H       = 320, 240
    BS         = 300
    D_FROM     = 200
    D_UNTIL    = 400
    D_INTERVAL = 50
    GRAD_THR   = 0.0005

# ── Apply to config ───────────────────────────────────────────────────────────
cfg = load_config(CONFIG_PATH)
cfg.training.iterations             = ITERS
cfg.renderer.max_gaussians          = MAX_GAUSS
cfg.renderer.sh_degree              = SH
cfg.training.output_dir             = OUTPUT_DIR
cfg.training.checkpoint_dir         = CKPT_DIR
cfg.training.densify_from_iter      = D_FROM
cfg.training.densify_until_iter     = D_UNTIL
cfg.training.densification_interval = D_INTERVAL
cfg.training.densify_grad_threshold = GRAD_THR

# Guard: cfg.viewer may not exist if viewer: block is absent from config.yaml
if not hasattr(cfg, "viewer"):
    cfg.viewer = type("Config", (), {})()
cfg.viewer.window_width  = W
cfg.viewer.window_height = H

# Guard: save_every may be absent from config.yaml
save_every = getattr(cfg.training, "save_every", 500)

print(f"\nTraining config:")
print(f"  iterations    : {ITERS:,}")
print(f"  max_gaussians : {MAX_GAUSS:,}  ← hard cap prevents OOM")
print(f"  sh_degree     : {SH}")
print(f"  resolution    : {W}x{H}")
print(f"  batch_size    : {BS}")
print(f"  densify       : iter {D_FROM} → {D_UNTIL}, every {D_INTERVAL} iters")
print(f"  grad_threshold: {GRAD_THR}")
print(f"  save_every    : {save_every}")

# ── Load COLMAP ───────────────────────────────────────────────────────────────
print(f"\n📂  Loading COLMAP from: {COLMAP_DIR}")
cameras_colmap, images_colmap, points3d = load_colmap_model(COLMAP_DIR)

# ── Build training cameras + images ───────────────────────────────────────────
print(f"🖼️   Loading {len(images_colmap)} images at {W}x{H}...")
train_cameras, train_images, missing = [], [], []

for img_data in sorted(images_colmap.values(), key=lambda i: i.name):
    cam_data = cameras_colmap[img_data.camera_id]
    cam      = ViewerCamera.from_colmap(img_data, cam_data, W, H)
    img_path = Path(FRAMES_DIR) / img_data.name
    if not img_path.exists():
        img_path = Path(FRAMES_DIR) / Path(img_data.name).name
    if not img_path.exists():
        missing.append(img_data.name)
        continue
    img_np = np.array(
        Image.open(img_path).convert("RGB").resize((W, H), Image.LANCZOS),
        dtype=np.float32
    ) / 255.0
    train_cameras.append(cam)
    train_images.append(torch.from_numpy(img_np).permute(2, 0, 1).cpu())

if missing:
    print(f"⚠️  {len(missing)} frames missing — first 3: {missing[:3]}")
if not train_cameras:
    raise RuntimeError(f"No images loaded. Check FRAMES_DIR={FRAMES_DIR}")
print(f"✅  {len(train_cameras)} cameras, {len(points3d):,} 3D points")

# ── Subsample if initial cloud exceeds MAX_GAUSS ─────────────────────────────
xyzs = np.array([p.xyz for p in points3d.values()], dtype=np.float32)
rgbs = np.array([p.rgb for p in points3d.values()], dtype=np.float32) / 255.0

if len(xyzs) > MAX_GAUSS:
    idx  = np.random.choice(len(xyzs), MAX_GAUSS, replace=False)
    xyzs = xyzs[idx];  rgbs = rgbs[idx]
    print(f"⚠️  Initial cloud subsampled: {len(points3d):,} → {MAX_GAUSS:,} points")

# ── Init model ───────────────────────────────────────────────────────────────
device = "cuda"
model = GaussianModel(sh_degree=SH)
model = model.to(device)
model.create_from_points(xyzs, rgbs)

# ── Renderer ─────────────────────────────────────────────────────────────────
torch.cuda.empty_cache();  gc.collect()
torch.backends.cudnn.benchmark = True

renderer = GaussianRenderer(
    width=W, height=H,
    bg_color=cfg.renderer.background_color,
    device=device,
    batch_size=BS,
    use_gsplat=True,
)

# ── Trainer ─────────────────────────────────────────────────────────────────
trainer = GaussianTrainer(
    model=model,
    renderer=renderer.render_torch,
    train_cameras=train_cameras,
    train_images=train_images,
    cfg=cfg,
)

# ── VRAM estimate ───────────────────────────────────────────────────────────
bytes_per_gauss = (3 + 1 + 3 + 3 + 4 + 48) * 4
vram_est_mb     = len(model) * bytes_per_gauss / 1e6
free_vram_gb    = torch.cuda.mem_get_info()[0] / 1e9
total_vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n📊  VRAM estimate : ~{vram_est_mb:.0f} MB for {len(model):,} Gaussians")
print(f"    Free VRAM now : {free_vram_gb:.1f} GB / {total_vram_gb:.1f} GB total")
if vram_est_mb > free_vram_gb * 1000 * 0.8:
    print("⚠️  Close to VRAM limit — lower MAX_GAUSS if you get OOM")

# ── Training loop ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  MonoSplat — OOM-safe Training")
print(f"  Job : {JOB_ID}  |  GPU : {torch.cuda.get_device_name(0)}")
print(f"  {ITERS:,} iters  |  max {MAX_GAUSS:,} Gaussians")
print(f"{'='*60}\n")

optimizer     = trainer.optimizer
n             = len(model)
grad_accum    = torch.zeros(n, device=device)
grad_denom    = torch.zeros(n, device=device)
nan_count     = 0
oom_count     = 0
loss_val      = 0.0

pbar = tqdm(range(ITERS), desc="Training", dynamic_ncols=True)

for it in pbar:
    idx    = np.random.randint(len(train_cameras))
    camera = train_cameras[idx]
    gt_img = train_images[idx].to(device, non_blocking=True)

    try:
        optimizer.zero_grad(set_to_none=True)
        rendered = renderer.render_torch(model, camera)
        loss     = combined_loss(rendered, gt_img, lambda_ssim=getattr(cfg.training, "lambda_dssim", 0.2))

        if torch.isnan(loss):
            nan_count += 1
            if nan_count <= 5 or nan_count % 50 == 0:
                tqdm.write(f"[{it:5d}]   NaN loss (total {nan_count}), skipping")
            continue

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        if model._positions.grad is not None:
            with torch.no_grad():
                gn = model._positions.grad.norm(dim=1)
                if gn.shape[0] == grad_accum.shape[0]:
                    grad_accum += gn
                    grad_denom += 1.0

        optimizer.step()
        trainer._apply_position_lr_decay(it, ITERS)
        loss_val = loss.item()

    except torch.cuda.OutOfMemoryError:
        oom_count += 1
        torch.cuda.empty_cache();  gc.collect()
        tqdm.write(f"[{it:5d}] OOM #{oom_count} — emergency prune to 60% cap")
        emergency = int(MAX_GAUSS * 0.6)
        cur = len(model)
        if cur > emergency:
            n_prune = cur - emergency
            with torch.no_grad():
                _, low = model.opacities.squeeze().topk(n_prune, largest=False)
                mask   = torch.zeros(cur, dtype=torch.bool, device=device)
                mask[low] = True
            model.prune_points(mask)
            trainer._setup_optimizer()
            optimizer  = trainer.optimizer
            n          = len(model)
            grad_accum = torch.zeros(n, device=device)
            grad_denom = torch.zeros(n, device=device)
        continue

    # SH degree scheduling
    if it % 1000 == 0:
        model.oneup_sh_degree()

    # Densification
    if D_FROM <= it <= D_UNTIL and it % D_INTERVAL == 0:
        trainer._grad_accum = grad_accum.clone()
        trainer._grad_denom = grad_denom.clone()
        trainer._densify_and_prune(grad_threshold=GRAD_THR, percent_dense=getattr(cfg.training, "percent_dense", 0.01))
        optimizer  = trainer.optimizer
        n          = len(model)
        grad_accum = torch.zeros(n, device=device)
        grad_denom = torch.zeros(n, device=device)
        tqdm.write(f"[{it:5d}] Densified -> {n:,} Gaussians")

    # Checkpoint
    if it > 0 and it % save_every == 0:
        trainer._save(it)

    # VRAM periodic clear
    if it % 500 == 0:
        torch.cuda.empty_cache()

    pbar.set_postfix({
        "loss": f"{loss_val:.4f}",
        "N":    f"{len(model):,}",
        "NaN":  nan_count,
        "OOM":  oom_count,
    })

trainer._save(ITERS)

print(f"\n✅  Training complete.")
print(f"   Gaussians   : {len(model):,}")
print(f"   Final loss  : {loss_val:.4f}")
print(f"   NaN skipped : {nan_count}")
print(f"   OOM events  : {oom_count}")
print(f"   Output dir  : {OUTPUT_DIR}")


In [ ]:
# Patch gaussian_model._knn_mean_dist in-place to use sampling+chunking
from pathlib import Path
import re
gm_path = Path('/content/monosplat/src/reconstruction/gaussian_model.py')
if not gm_path.exists():
    raise FileNotFoundError(f'Expected gaussian_model.py at {gm_path} — check repo layout')
src = gm_path.read_text()
# Replace the existing function by searching the def and the following def _build_rotation_matrix
pattern = r'def _knn_mean_dist\([^:]*:\n(?:.|\n)*?\n\s*def _build_rotation_matrix'
m = re.search(pattern, src)
if not m:
    print('Could not locate existing _knn_mean_dist definition — aborting patch')
else:
    before = src[:m.start()]
    after = src[m.end()-len('\n    def _build_rotation_matrix'): ]
    new_fn = '''def _knn_mean_dist(pts: torch.Tensor, k: int = 3, max_sample: int = 8192, chunk_size: int = 4096) -> torch.Tensor:
    """Mean distance to k nearest neighbours.

    For large point clouds this avoids allocating the full N x N distance
    matrix by sampling a subset of points and assigning each original
    point the mean-k distance of its nearest sampled neighbour. This
    provides a good approximation of local density while keeping memory
    use bounded.
    """
    N = pts.shape[0]
    if N == 0:
        return torch.tensor([], device=pts.device)
    k_actual = min(k, N - 1)
    if k_actual == 0:
        return torch.zeros(N, device=pts.device)

    device = pts.device

    # If the point cloud is small enough, compute exact pairwise distances
    if N <= max_sample:
        dist_matrix = torch.cdist(pts, pts)
        dist_matrix.fill_diagonal_(float('inf'))
        knn_dists = torch.topk(dist_matrix, k=k_actual, largest=False, dim=1).values
        return knn_dists.mean(dim=1).clamp(min=1e-7)

    # Otherwise: sample a subset, compute mean-k for the sample, then
    # assign to all points the value of their nearest sample neighbour.
    perm = torch.randperm(N, device=device)[:max_sample]
    pts_sample = pts[perm]

    dist_sample = torch.cdist(pts_sample, pts_sample)
    dist_sample.fill_diagonal_(float('inf'))
    knn_sample = torch.topk(dist_sample, k=k_actual, largest=False, dim=1).values
    mean_sample = knn_sample.mean(dim=1).clamp(min=1e-7)  # (M,)

    # For memory safety, compute distances from full set to sample in chunks
    mean_all = torch.empty(N, device=device, dtype=mean_sample.dtype)
    for i in range(0, N, chunk_size):
        j = min(i + chunk_size, N)
        d = torch.cdist(pts[i:j], pts_sample)  # (chunk, M)
        idx = d.argmin(dim=1)
        mean_all[i:j] = mean_sample[idx]

    return mean_all

    def _build_rotation_matrix(rotations: torch.Tensor) -> torch.Tensor:'''
    new_src = before + new_fn + after
    gm_path.write_text(new_src)
    print('Patched', gm_path)


In [ ]:
# Import check with automatic pip install for missing packages (best-effort)
import importlib, subprocess, sys

def try_import(name, pip_name=None):
    try:
        return importlib.import_module(name)
    except Exception as e:
        pkg = pip_name or name
        print(f"Missing {name}; attempting pip install {pkg}...")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
        return importlib.import_module(name)

print('Testing core imports from repo...')
sys.path.insert(0, '/content/monosplat')
try:
    from src.reconstruction.gaussian_model import GaussianModel, _knn_mean_dist
    from src.reconstruction.trainer import GaussianTrainer
    from src.reconstruction.loss import combined_loss
    print('✅ Core imports successful')
except Exception as e:
    print('Import error — details:')
    print(e)
    raise


In [ ]:
# Quick smoke test for the patched _knn_mean_dist (synthetic data)
import torch
from src.reconstruction.gaussian_model import _knn_mean_dist

pts = torch.randn(20000, 3)  # moderate size — sampling will be used
d = _knn_mean_dist(pts, k=3, max_sample=2048, chunk_size=4096)
print('smoke result shape:', d.shape)
print('sample values:', d[:6])


## Next steps

- If smoke test is successful, set runtime to GPU (Runtime → Change runtime type → GPU) and run your training cells.
- If you prefer kNN with FAISS for faster/accurate NN instead of the sampling approximation, reply and I will add a FAISS-backed function.
- If any ImportError appears after running the notebook, paste the error here and I'll provide the exact `pip install` command to fix it.